Step 1: Generate & Persist Raw Delta Tables
Run this in a Databricks Notebook cell. We generate the data in Pandas first for ease, then convert to Spark for Delta persistence.

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql.functions import *

# 1. Setup Data
n_rows = 2000
policy_ids = [f"POL_{i:04d}" for i in range(n_rows)]

# Policy Table
policy_pdf = pd.DataFrame({
    'policy_id': policy_ids,
    'age': np.random.randint(18, 75, n_rows),
    'policy_type': np.random.choice(['Basic', 'Premium', 'Platinum'], n_rows),
    'annual_premium': np.random.uniform(500, 3000, n_rows),
    'is_cancelled': np.random.choice([0, 1], n_rows, p=[0.8, 0.2])
})

# Claims Table (Aggregate later)
claims_pdf = pd.DataFrame({
    'policy_id': np.random.choice(policy_ids, 800),
    'claim_amount': np.random.uniform(100, 5000, 800)
})

# Billing Table
billing_pdf = pd.DataFrame({
    'policy_id': policy_ids,
    'late_payments': np.random.choice([0, 1, 2], n_rows, p=[0.8, 0.15, 0.05])
})

# 2. Persist to Delta
spark.createDataFrame(policy_pdf).write.mode("overwrite").saveAsTable("policy_raw")
spark.createDataFrame(claims_pdf).write.mode("overwrite").saveAsTable("claims_raw")
spark.createDataFrame(billing_pdf).write.mode("overwrite").saveAsTable("billing_raw")

Step 2: Feature Engineering & Transformation
We transform raw attributes into features (e.g., total claim costs) and join them.

In [0]:
# Create features from claims
from pyspark.sql.functions import count, sum
claims_feat = spark.table("claims_raw").groupBy("policy_id").agg(
    count("claim_amount").alias("total_claims"),
    sum("claim_amount").alias("total_claim_val")
)

# Combine into a final feature set
features_df = spark.table("policy_raw") \
    .join(spark.table("billing_raw"), "policy_id", "left") \
    .join(claims_feat, "policy_id", "left") \
    .fillna(0)

# Light transformation: convert policy_type to numeric index
from pyspark.ml.feature import StringIndexer
indexer = StringIndexer(inputCol="policy_type", outputCol="policy_type_idx")
final_features_df = indexer.fit(features_df).transform(features_df)

Step 3: Register in Databricks Feature Store
This allows other data scientists to discover these features and ensures "Point-in-Time" correctness.

In [0]:
%pip install databricks-feature-engineering
dbutils.library.restartPython()

In [0]:
from databricks.feature_store import FeatureStoreClient

fs = FeatureStoreClient()

fs.create_table(
    name="policy_at_risk_features",
    primary_keys=["policy_id"],
    df=final_features_df.drop("is_cancelled"),
    description="Features for predicting policy risk"
)

In [0]:
!pip install xgboost

In [0]:
final_features_df.display()

In [0]:
import mlflow
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from databricks.feature_store import FeatureLookup
import pandas as pd
import numpy as np
from pyspark.sql.functions import *


# Pull data from Feature Store
training_set = fs.create_training_set(
    df=final_features_df.select("policy_id", "is_cancelled"), # The label + ID
    feature_lookups=[
        FeatureLookup(
            table_name="policy_at_risk_features",
            lookup_key="policy_id"
        )
    ],
    label="is_cancelled"
)

df_ml = training_set.load_df().toPandas()
X = pd.get_dummies(df_ml.drop(["policy_id", "is_cancelled"], axis=1))
y = df_ml["is_cancelled"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Model 1: Logistic Regression ---
with mlflow.start_run(run_name="Linear_Model") as run1:
    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_train, y_train)
    mlflow.sklearn.log_model(lr, "model")
    mlflow.log_metric("accuracy", lr.score(X_test, y_test))

# --- Model 2: XGBoost ---
with mlflow.start_run(run_name="XGBoost_Model") as run2:
    xgb = XGBClassifier(use_label_encoder=False, eval_metric="logloss")
    xgb.fit(X_train, y_train)
    mlflow.xgboost.log_model(xgb, "model")
    mlflow.log_metric("accuracy", xgb.score(X_test, y_test))